# nanolab — one-click training run (Qwen3-0.6B on gsm8k)

**Setup once:** GPU on (Colab: *Runtime → Change runtime type → T4* · Kaggle:
*Settings → Accelerator → GPU T4* + **Internet ON**).

**Then just Run All.** Every cell is safe to re-run: training auto-resumes
from the last checkpoint (or starts fresh), the exam finds the newest
checkpoint by itself, and artifacts are zipped at the end.

**Zero-babysitting option (Kaggle):** *Save Version → Save & Run All
(Commit)* runs the whole notebook in the background — close the tab, come
back later, download the zip from the version's Output tab.

In [ ]:
# 1) Setup: code, Drive persistence (Colab), deps, environment, sanity tests
import os
!rm -rf nanolab && git clone -q https://github.com/khwahish1509/RLPost.git nanolab
%cd nanolab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base = '/content/drive/MyDrive/nanolab'
    os.makedirs(f'{base}/adapters', exist_ok=True)
    os.makedirs(f'{base}/results', exist_ok=True)
    !rm -rf adapters results && ln -s {base}/adapters adapters && ln -s {base}/results results
    print('persistence: Google Drive (MyDrive/nanolab)')
except ImportError:
    print('persistence: Kaggle working dir / version output')
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv sync -q && uv add -q torch transformers peft && uv tool install -q prime
!uv run python -c "import torch; print('CUDA available:', torch.cuda.is_available())"
!uv run nanolab env install primeintellect/gsm8k
!uv run pytest -q

In [ ]:
# 2) Train — auto-resumes from the last checkpoint, or starts fresh
!uv run nanolab train configs/qwen3-0.6b-gsm8k.toml --resume

In [ ]:
# 3) The exam: base vs newest checkpoint, 64 held-out questions, greedy
!uv run python scripts/compare_adapter.py

In [ ]:
# 4) Package artifacts (adapters + lab db). Colab: browser download.
#    Kaggle: appears in the notebook/version Output.
!zip -qr ../nanolab-artifacts.zip adapters/ results/nanolab.db
try:
    from google.colab import files
    files.download('../nanolab-artifacts.zip')
except ImportError:
    print('nanolab-artifacts.zip written one directory up (see Output panel)')